In [ ]:
!pip install transformers==4.40.2 peft==0.10.0 accelerate==0.30.1 trl==0.8.6

In [ ]:
# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

# Hugging Face
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    AutoModel,
    AutoTokenizer,
    Trainer,
    TrainingArguments
)

# Dataset
from datasets import load_dataset

# Utilities
from torch.utils.data import DataLoader
from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

In [ ]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2-large")

# GPT-2 fix
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def format_squad(example):
    context = example["context"][:300]
    question = example["question"]
    answer = example["answers"]["text"][0]

    return {
        "text": f"""You are a helpful assistant.

Context: {context}

Question: {question}

Answer: {answer}"""
    }

In [ ]:
from datasets import load_dataset

dataset = load_dataset("squad")

train_data = dataset["train"].map(format_squad)

In [ ]:
def tokenize(example):
    outputs = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

In [ ]:
tokenized_dataset = train_data.map(
    tokenize,
    batched=True,
    remove_columns=train_data.column_names
)

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained("gpt2-medium")


model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./sft_model",

    # Core
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    max_steps=625,

    learning_rate=2e-6,
    warmup_steps=100,
    lr_scheduler_type="cosine",

    evaluation_strategy="no",
    eval_steps=100,
    logging_steps=50,
    save_steps=200,

    load_best_model_at_end=False,

    # Memory
    fp16=True,
    gradient_checkpointing=True,
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

In [ ]:
trainer.train()

In [ ]:
logs = trainer.state.log_history

In [ ]:
steps = []
losses = []

for log in logs:
    if "loss" in log:
        steps.append(log["step"])
        losses.append(log["loss"])

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(steps, losses)
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("SFT Training Loss Curve")
plt.show()

In [ ]:
try:
    trainer.save_model("./sft_model_final")
    tokenizer.save_pretrained("./sft_model_final")
    print("Saved!")
except Exception as e:
    print("Error:", e)\

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

In [ ]:
model_name = "gpt2-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
base_model = AutoModel.from_pretrained("gpt2-medium")

In [ ]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
class RewardModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model

        self.reward_head = nn.Sequential(
            nn.Linear(base_model.config.hidden_size, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def mean_pool(self, hidden_states, mask):
        mask = mask.unsqueeze(-1)
        return (hidden_states * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        hidden = outputs.last_hidden_state
        pooled = self.mean_pool(hidden, attention_mask)

        reward = self.reward_head(pooled)
        return reward

In [ ]:
dataset = load_dataset("Anthropic/hh-rlhf", split="train[:3%]")

In [ ]:
def preprocess(example):
    chosen = tokenizer(
        example["chosen"],
        truncation=True,
        max_length=128
    )

    rejected = tokenizer(
        example["rejected"],
        truncation=True,
        max_length=128
    )

    return {
        "chosen_input_ids": chosen["input_ids"],
        "chosen_attention_mask": chosen["attention_mask"],
        "rejected_input_ids": rejected["input_ids"],
        "rejected_attention_mask": rejected["attention_mask"],
    }

dataset = dataset.map(preprocess)
dataset.set_format(type="torch")

In [ ]:
def reward_collator(batch):
    chosen = {
        "input_ids": [item["chosen_input_ids"] for item in batch],
        "attention_mask": [item["chosen_attention_mask"] for item in batch],
    }

    rejected = {
        "input_ids": [item["rejected_input_ids"] for item in batch],
        "attention_mask": [item["rejected_attention_mask"] for item in batch],
    }

    chosen_padded = tokenizer.pad(chosen, return_tensors="pt")
    rejected_padded = tokenizer.pad(rejected, return_tensors="pt")

    return {
        "chosen_input_ids": chosen_padded["input_ids"],
        "chosen_attention_mask": chosen_padded["attention_mask"],
        "rejected_input_ids": rejected_padded["input_ids"],
        "rejected_attention_mask": rejected_padded["attention_mask"],
    }

In [ ]:
dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=reward_collator
)

In [ ]:
def reward_loss(r_chosen, r_rejected):
    return F.softplus(r_rejected - r_chosen).mean()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RewardModel(base_model).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

epochs = 2

In [ ]:
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in dataloader:
        optimizer.zero_grad()

        chosen_ids = batch["chosen_input_ids"].to(device)
        chosen_mask = batch["chosen_attention_mask"].to(device)

        rejected_ids = batch["rejected_input_ids"].to(device)
        rejected_mask = batch["rejected_attention_mask"].to(device)

        r_chosen = model(chosen_ids, chosen_mask)
        r_rejected = model(rejected_ids, rejected_mask)

        loss = reward_loss(r_chosen, r_rejected)

        loss.backward()

        # stability trick
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Avg Loss: {total_loss / len(dataloader):.4f}")

In [ ]:
losses = []
accuracies = []

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in dataloader:
        optimizer.zero_grad()

        chosen_ids = batch["chosen_input_ids"].to(device)
        chosen_mask = batch["chosen_attention_mask"].to(device)

        rejected_ids = batch["rejected_input_ids"].to(device)
        rejected_mask = batch["rejected_attention_mask"].to(device)

        r_chosen = model(chosen_ids, chosen_mask)
        r_rejected = model(rejected_ids, rejected_mask)

        loss = reward_loss(r_chosen, r_rejected)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        # accuracy
        correct += (r_chosen > r_rejected).sum().item()
        total += r_chosen.size(0)

    avg_loss = total_loss / len(dataloader)
    acc = correct / total

    losses.append(avg_loss)
    accuracies.append(acc)

    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Accuracy: {acc:.4f}")

In [ ]:
import os
import torch

save_dir = "./reward_model"
os.makedirs(save_dir, exist_ok=True)

print("Saving to:", os.path.abspath(save_dir))

# Try safe saving
try:
    model.save_pretrained(save_dir)
    print("✅ Base model saved")
except Exception as e:
    print("❌ Base model error:", e)

# Reward head
try:
    if hasattr(model, "reward_head"):
        torch.save(model.reward_head.state_dict(), f"{save_dir}/reward_head.pt")
        print("✅ Reward head saved")
    else:
        print("⚠️ No reward_head found")
except Exception as e:
    print("❌ Reward head error:", e)

# Tokenizer
try:
    tokenizer.save_pretrained(save_dir)
    print("✅ Tokenizer saved")
except Exception as e:
    print("❌ Tokenizer error:", e)

# Final check
print("Files in dir:", os.listdir(save_dir))